In [31]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
docs = PyPDFDirectoryLoader("papers/").load()
print(len(docs))  # one Document per page, with metadata (source, page)



fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/ETTOVJ+CMR8', '/FontDescriptor': IndirectObject(48, 0, 2224214403920), '/Type': '/Font', '/FirstChar': 46, '/LastChar': 115, '/Widths': [295, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 826, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 531, 0, 472, 0, 0, 0, 0, 590, 0, 0, 0, 0, 885, 590, 531, 590, 0, 0, 419], '/Encoding': '/WinAnsiEncoding', '/Subtype': '/Type1'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/LHEAZV+CMSY6', '/FontDescriptor': IndirectObject(45, 0, 2224214403920), '/Type': '/Font', '/FirstChar': 3, '/LastChar': 3, '/Widths': [638], '/Encoding': IndirectObject(199, 0, 2224214403920), '/Subtype': '/Type1'}, but is not installed. Consider installing fontTools if you enc

1631


In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(docs)
#save chunks to disk for later use
import os, json

for i, chunk in enumerate(chunks):
    with open(f"testPypdf_chunks/chunk_{i}.json", "w") as f:
        json.dump(chunk.dict(), f)  

C:\Users\Zenbook\AppData\Local\Temp\ipykernel_14968\2201428692.py:9: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  json.dump(chunk.dict(), f)


In [30]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
db = Chroma.from_documents(chunks, embeddings, persist_directory="chroma_db")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 12074.07it/s]


TypeError: TextEncodeInput must be Union[TextInputSequence, Tuple[InputSequence, InputSequence]]

In [26]:
retriever = db.as_retriever(search_kwargs={"k": 5})
results = retriever.invoke("How to calculate the ground state energy of a quantum system using VQE?")
for r in results:
    print(r.metadata["source"], r.metadata["page"], r.page_content[:200])

papers\2512.22572v2.pdf 0 tion. At present, we find ourselves in the era of Noisy
Intermediate-Scale Quantum (NISQ) [2] devices, mean-
ing that quantum devices are built from a few hundred
qubits and suffer from qubit errors (
papers\2512.22572v2.pdf 0 Variational quantum eigensolver for chemical molecules
Luca Ion and Adam Smith
School of Physics & Astronomy, University of Nottingham, Nottingham, UK
(Dated: May 16, 2023)
Solving multi-particle syst
papers\2610.29779v1.pdf 0 accelerate molecular simulations [4].
Among the most promising approaches for near-term
quantum devices are Variational Quantum Algorithms
(VQAs)[5], particularly the Variational Quantum Eigen-
solver
papers\2607.02620v1.pdf 1 electronic wavefunctions [9], qubits are capable of representing these systems and then approximating the
ground state of the Hamiltonian variationally using a parameterized trial state [10].
In the c
papers\2611.16610v1.pdf 2 2.2 Variational Quantum Eigensolver
The Variational Quantum Eigens

In [23]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

llm = ChatOpenAI(
    model="crow-uncensored:free",
    base_url="https://crowllm.com/v1",
    api_key=os.getenv("CROW_API_KEY"),
    temperature=0,
)

print(llm.invoke("Say hello in one sentence.").content)

UncensoredAI
Hello.


In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    """Answer using ONLY the context below. Cite sources as [source, page].
If the answer is not in the context, say "Not found in the sources."

Context:
{context}

Question: {question}"""
)

def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata['source']}, p.{d.metadata['page']}] {d.page_content}" for d in docs
    )

chain = (
    {"context": retriever | format_docs, "question": lambda x: x}
    | prompt | llm | StrOutputParser()
)

print(chain.invoke("Can you explain the mathematical intuition behind a hamiltonian ?"))

UncensoredAI

text
Not found in the sources.


In [25]:
print(chain.invoke("From where does a hamiltonin come from in quantum mechanics?"))

UncensoredAI
Not found in the sources.
